In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
np.random.seed(42)  # để kết quả tái lập được

# ============================================================
# 1. TẠO DỮ LIỆU GIẢ LẬP (MOCK DATA) CHO NHIỀU NGÀY, MỖI NGÀY 24 GIỜ
# ============================================================
N_DAYS = 30
HOURS_PER_DAY = 24
total_hours = N_DAYS * HOURS_PER_DAY
hours = np.arange(total_hours)
hour_of_day = hours % HOURS_PER_DAY
phase = 2 * np.pi * (hour_of_day - 5) / 24

# Nhiệt độ: đáy ~5h, đỉnh ~14h (dùng 2 hài hòa để lệch pha đúng)
temperature = 25 - 5 * np.cos(phase) - 1.2 * np.cos(2 * phase - np.pi / 2)
temperature += np.random.normal(0, 0.4, size=total_hours)

# Độ ẩm không khí: ngược chiều với nhiệt độ
humidity = 80 + 15 * np.cos(phase) + 3.6 * np.cos(2 * phase - np.pi / 2)
humidity += np.random.normal(0, 1.0, size=total_hours)
humidity = np.clip(humidity, 0, 100)

# Ánh sáng (Lux): chỉ có mặt trời từ 6h đến 18h, đỉnh lúc 12h trưa
light_lux = np.where(
    (hour_of_day >= 6) & (hour_of_day <= 18),
    50000 * np.sin(np.pi * (hour_of_day - 6) / 12),
    0
)
light_lux = light_lux + np.random.normal(0, 300, size=total_hours)
light_lux = np.clip(light_lux, 0, None)

# Độ ẩm đất: giảm dần trong ngày, hồi phục mỗi đêm, bốc hơi mạnh hơn khi nắng gắt
soil_moisture = np.zeros(total_hours)
soil_moisture[0] = 60
for i in range(1, total_hours):
    if hour_of_day[i] == 0:
        recovery = np.random.uniform(15, 25)
        soil_moisture[i] = min(70, soil_moisture[i - 1] + recovery)
    else:
        evap = 0.3 + (light_lux[i] / 10000) * 0.5
        soil_moisture[i] = soil_moisture[i - 1] - evap
soil_moisture = np.clip(soil_moisture, 0, 100)
soil_moisture += np.random.normal(0, 0.3, size=total_hours)

df = pd.DataFrame({
    'Hour': hours,
    'Hour_of_day': hour_of_day,
    'Temperature_C': temperature,
    'Humidity_%': humidity,
    'Light_Lux': light_lux,
    'Soil_Moisture_%': soil_moisture
})

print("--- DỮ LIỆU CẢM BIẾN MÔ PHỎNG (5 GIỜ ĐẦU) ---")
print(df.head())
print(f"\nTổng số mẫu: {len(df)} giờ ({N_DAYS} ngày)")

# ============================================================
# 2. CHUẨN HÓA DỮ LIỆU (MIN-MAX SCALING) CHO MẠNG LSTM
# ============================================================
train_size = int(len(df) * 0.8)
feature_cols = ['Temperature_C', 'Humidity_%', 'Light_Lux', 'Soil_Moisture_%']
TARGET_COL_INDEX = feature_cols.index('Soil_Moisture_%')  # = 3, cột mục tiêu duy nhất

scaler = MinMaxScaler()
scaler.fit(df[feature_cols].iloc[:train_size])  # chỉ fit trên tập train
scaled_data = scaler.transform(df[feature_cols])
scaled_df = pd.DataFrame(scaled_data, columns=['Temp_Scaled', 'Humid_Scaled', 'Light_Scaled', 'Soil_Scaled'])

print("\n--- DỮ LIỆU SAU KHI CHUẨN HÓA [0, 1] (FLOAT32, DÙNG ĐỂ TRAIN) ---")
print(scaled_df.head())

# ============================================================
# FIX #1 + #2: create_sequences() giờ trả về y CHỈ 1 CỘT (Soil_Moisture_%)
# => output shape đúng [Batch_size, 1] theo bản đặc tả
# horizon mặc định đổi thành 6 (dự báo 6 giờ tới) thay vì 1
# ============================================================
def create_sequences(data: np.ndarray, target_col_index: int, window_size: int = 24, horizon: int = 6):
    """
    Cắt dữ liệu thành các cửa sổ trượt để train LSTM.
    - data: mảng 2D shape (n_samples, n_features), đã scale, X dùng ĐỦ 4 cột (đa biến đầu vào).
    - target_col_index: chỉ số cột dùng làm nhãn y (ở đây là Soil_Moisture_%, index 3).
    - window_size: số bước thời gian đầu vào (24 = nhìn lại 24h).
    - horizon: dự báo bao xa vào tương lai (6 = 6 giờ tới, phục vụ chiến lược tưới vi lượng).
    Trả về:
        X shape (n, window_size, n_features)  -- đầu vào vẫn đa biến, đủ 4 cảm biến
        y shape (n, 1)                        -- ĐÚNG THEO SPEC: chỉ dự đoán Soil_Moisture_%
    """
    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i: i + window_size])
        y.append(data[i + window_size + horizon - 1, target_col_index])  # chỉ lấy 1 cột mục tiêu
    X = np.array(X)
    y = np.array(y).reshape(-1, 1)  # ép rõ shape (n, 1) thay vì (n,) hoặc (n, 4)
    return X, y

WINDOW_SIZE = 24
HORIZON = 6  # FIX #2: dự báo 6 giờ tới (có thể đổi thành 12 nếu muốn xa hơn)

X_all, y_all = create_sequences(scaled_data, target_col_index=TARGET_COL_INDEX,
                                 window_size=WINDOW_SIZE, horizon=HORIZON)

split_idx = train_size - WINDOW_SIZE - HORIZON + 1
X_train, y_train = X_all[:split_idx], y_all[:split_idx]
X_test, y_test = X_all[split_idx:], y_all[split_idx:]

print(f"\n--- SEQUENCES CHO LSTM (horizon={HORIZON}h) ---")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}   <-- y phải là (N, 1)")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")
assert y_train.shape[1] == 1, "Output shape sai — phải là [Batch_size, 1] theo bản đặc tả"






--- DỮ LIỆU CẢM BIẾN MÔ PHỎNG (5 GIỜ ĐẦU) ---
   Hour  Hour_of_day  Temperature_C  Humidity_%   Light_Lux  Soil_Moisture_%
0     0            0      24.504590   82.390087  198.864381        59.830890
1     1            1      23.483925   82.672140  352.042157        59.657488
2     2            2      22.923542   85.658416   54.306468        59.319458
3     3            3      22.318315   90.615954    0.000000        58.710040
4     4            4      20.676710   92.859753  119.906386        58.908046

Tổng số mẫu: 720 giờ (30 ngày)

--- DỮ LIỆU SAU KHI CHUẨN HÓA [0, 1] (FLOAT32, DÙNG ĐỂ TRAIN) ---
   Temp_Scaled  Humid_Scaled  Light_Scaled  Soil_Scaled
0     0.446184      0.557263      0.003939     1.000000
1     0.365851      0.564973      0.006973     0.997140
2     0.321745      0.646598      0.001076     0.991566
3     0.274110      0.782103      0.000000     0.981515
4     0.144906      0.843434      0.002375     0.984781

--- SEQUENCES CHO LSTM (horizon=6h) ---
X_train: (547, 2